# Using the DynamicMethod Class

## Introduction

DynamicMethod is an abstract method wrapper with multiplexed binding and callback. It inherits from DynamicCallable and BaseMethod, and overrides __call__ to automatically pass the bound instance self._self_() to the selected call strategy. It is optimized for instance methods.

This tutorial emphasizes method multiplexer functionality for binding and callback.

### Table of Contents
- Importing
- Core behavior
- Binding with MethodMultiplexer
- Calling with MethodMultiplexer (auto self)
- Examples
- FAQs


## Importing

In [ ]:
from baseobjects.functions import DynamicMethod
from baseobjects.functions import MethodMultiplexer


## Core Behavior

- default_bind_method = "bind_self"
- default_call_method = "call_wrapped"
- __call__ delegates to call_multiplexer but injects self._self_() as the first argument, ensuring proper bound-method semantics.
- bind_method and call_method properties select strategies inside the respective MethodMultiplexer.


## Binding with MethodMultiplexer

When used as a descriptor on a class, accessing through an instance triggers __get__, which delegates to bind_multiplexer. With default_bind_method="bind_self", the result acts like a bound method wrapper.

In [ ]:
class Service:
    def __init__(self, name):
        self.name = name

# Function-style callable expecting (self, x)
def shout(self, x):
    return f"{self.name}! {x.upper()}"

Service.say = DynamicMethod(shout)

svc = Service('Echo')
print(svc.say('hello'))  # binds self via bind_multiplexer

# Optionally change binding behavior
Service.say.bind_method = 'bind_self'  # remains default
print(svc.say('world'))


## Calling with MethodMultiplexer (auto self)

DynamicMethod.__call__ is implemented as:
- return self.call_multiplexer(self._self_(), *args, **kwargs)

This means the selected call strategy is invoked with the instance automatically provided.

In [ ]:
class VerboseMethod(DynamicMethod):
    def call_wrapped(self, *args, **kwargs):
        return self._wrapped_(*args, **kwargs)
    
    def call_logged(self, *args, **kwargs):
        result = self._wrapped_(*args, **kwargs)
        print(f"call_logged(self={args[0]!r}, args={args[1:]}, result={result})")
        return result

Service.shout = VerboseMethod(shout)
print(svc.shout('hi'))  # default call_wrapped

Service.shout.call_method = 'call_logged'
print(svc.shout('there'))


## Examples

### Example 1: Strategy switching on instance methods

In [ ]:
class Math:
    def __init__(self, name='M'):
        self.name = name

# Operation expecting (self, a, b)
def add(self, a, b):
    return a + b

def mul(self, a, b):
    return a * b

Math.op = DynamicMethod(add)

m = Math()
print('add:', m.op(2,3))

# Switch wrapped function by reassigning
Math.op = DynamicMethod(mul)
print('mul:', m.op(2,3))

# Or keep same wrapped function but switch call strategy
class CheckedMethod(DynamicMethod):
    def call_wrapped(self, *args, **kwargs):
        return self._wrapped_(*args, **kwargs)
    def call_checked(self, *args, **kwargs):
        a, b = args[1], args[2]
        if b == 0:
            return 'division by zero not allowed'
        return self._wrapped_(*args, **kwargs)

def div(self, a, b):
    return a / b

Math.safe_div = CheckedMethod(div)
print('div:', m.safe_div(10,2))

Math.safe_div.call_method = 'call_checked'
print('checked div:', m.safe_div(10,0))


## FAQs

Q: Why DynamicMethod instead of DynamicFunction for methods?\n
A: DynamicMethod automatically injects the instance when calling via __call__, and defaults to bind_self for descriptor binding. This makes it the correct choice for method-like callables.

Q: How does MethodMultiplexer influence behavior?\n
A: It selects which binding and call strategies to use. You can switch strategies by setting bind_method and call_method on the DynamicMethod instance.
